# YachidaS 2019 CRC: SPECTRA analysis

Normal BMI range: $18.5 \leq BMI < 25$.

The input is a relative-abundance matrix with samples in rows and MetaPhlAn3 feature names in columns. SPECTRA performs preprocessing internally.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

WORK_DIR = Path.cwd()
if not (WORK_DIR / '4. Name-converted relative abundance.csv').exists():
    raise FileNotFoundError('Run this notebook from its own directory.')

PROJECT_ROOT = next(
    path for path in (WORK_DIR, *WORK_DIR.parents)
    if (path / 'SPECTRA_GitHub_resource').exists()
)
utils_dir = PROJECT_ROOT / 'SPECTRA_GitHub_resource/scripts'
if str(utils_dir) not in sys.path:
    sys.path.insert(0, str(utils_dir))

from utils import predict_from_abundance_to_phenotype


## 1. Relative-abundance input

In [2]:
abundance = pd.read_csv(WORK_DIR / '4. Name-converted relative abundance.csv', index_col=0, float_precision='round_trip')
metadata = pd.read_csv(WORK_DIR / '0. metadata.csv', index_col=0, float_precision='round_trip')
metadata.index = metadata.index.astype(str)
abundance.index = abundance.index.astype(str)
metadata = metadata.loc[abundance.index]

assert abundance.shape[0] == 417
assert metadata['BMI'].ge(18.5).all() and metadata['BMI'].lt(25).all()
assert metadata['true_label'].value_counts().to_dict() == {'HC': 235, 'CL': 182}

pd.DataFrame({
    'value': [abundance.shape[0], abundance.shape[1], 235, 182]
}, index=['samples', 'features', 'HC', 'CL'])

,value
samples,417
features,718
HC,235
CL,182


## 2. SPECTRA prediction

In [3]:
model_dir = PROJECT_ROOT / 'SPECTRA_GitHub_resource/models/metagenomic'
result = predict_from_abundance_to_phenotype(
    Abundance=abundance,
    MRI_model_path=model_dir / 'mri_calculators',
    SPECTRA_model_path=model_dir / 'spectra_model.pkl',
)

mri_score = result['MRI'].reindex(abundance.index)
probability = result['probability'].reindex(abundance.index)
mri_score.to_csv(WORK_DIR / '5. MRI scores.csv', float_format='%.17g')
probability.to_csv(WORK_DIR / '6. Probability.csv', float_format='%.17g')

display(mri_score.head())
display(probability.head())

,ACVD,AS,BloodPressureAbnormalities,ColorectalLesions,IBD,IGT,T2D,cirrhosis,control,fatty_liver,melanoma,schizophrenia
sample_id,,,,,,,,,,,,
SAMD00114718,0.11,0.01,0.00,0.32,0.21,0.00,0.02,0.02,0.53,0.0,0.0,0.0
SAMD00114719,0.00,0.05,0.00,0.00,0.01,0.00,0.00,0.07,0.86,0.0,0.0,0.0
SAMD00114720,0.01,0.06,0.06,0.16,0.06,0.02,0.00,0.00,0.72,0.0,0.0,0.0
SAMD00114722,0.00,0.00,0.00,1.00,0.02,0.01,0.00,0.00,0.04,0.0,0.0,0.0
SAMD00114723,0.02,0.01,0.00,0.93,0.03,0.00,0.00,0.00,0.15,0.0,0.0,0.0


,ACVD,AS,BPA,CL,IBD,IGT,T2D,CI,HC,FL,ME,SC
sample_id,,,,,,,,,,,,
SAMD00114718,0.026925,0.017091,0.007117,0.173101,0.121756,0.006124,0.026342,0.012317,0.586901,0.012742,0.008064,0.001520
SAMD00114719,0.000000,0.000812,0.001556,0.006302,0.005401,0.017862,0.000238,0.000000,0.960386,0.000000,0.004488,0.002957
SAMD00114720,0.004646,0.004056,0.005666,0.036917,0.034959,0.001931,0.024578,0.003925,0.867801,0.003761,0.007382,0.004377
SAMD00114722,0.001508,0.000000,0.000000,0.962232,0.001730,0.000000,0.001246,0.000000,0.030741,0.002542,0.000000,0.000000
SAMD00114723,0.002168,0.000000,0.000000,0.964932,0.000227,0.000000,0.002660,0.000000,0.026365,0.002543,0.001106,0.000000


## 3. Performance evaluation

In [4]:
case = metadata['true_label'].eq('CL').astype(int)
spectra_auc = roc_auc_score(case, probability['CL'])
pd.DataFrame({'AUC': [spectra_auc]}, index=['SPECTRA CL probability'])

,AUC
SPECTRA CL probability,0.879507


In [5]:
ranked_labels = np.argsort(-probability.to_numpy(), axis=1)[:, :3]
ranked_labels = probability.columns.to_numpy()[ranked_labels]

per_sample = metadata[['BMI', 'disease', 'true_label']].copy()
for rank in range(3):
    per_sample[f'Rank{rank + 1} label'] = ranked_labels[:, rank]
for k in range(1, 4):
    per_sample[f'Top{k} correct'] = [
        true_label in labels[:k]
        for true_label, labels in zip(per_sample['true_label'], ranked_labels)
    ]
rows = []
for scope, mask in {
    'CL patients': per_sample['true_label'].eq('CL'),
    'HC': per_sample['true_label'].eq('HC'),
    'All samples': pd.Series(True, index=per_sample.index),
}.items():
    row = {'scope': scope, 'n': int(mask.sum())}
    for k in range(1, 4):
        row[f'Top{k} accuracy'] = per_sample.loc[mask, f'Top{k} correct'].mean()
    rows.append(row)

top_accuracy = pd.DataFrame(rows).set_index('scope')
top_accuracy

,n,Top1 accuracy,Top2 accuracy,Top3 accuracy
scope,,,,
CL patients,182,0.846154,0.934066,0.983516
HC,235,0.774468,1.000000,1.000000
All samples,417,0.805755,0.971223,0.992806


In [6]:
summary = pd.DataFrame({
    'value': [
        spectra_auc,
        top_accuracy.loc['CL patients', 'Top1 accuracy'],
        top_accuracy.loc['CL patients', 'Top2 accuracy'],
        top_accuracy.loc['CL patients', 'Top3 accuracy'],
    ]
}, index=['SPECTRA AUC', 'CL Top1', 'CL Top2', 'CL Top3'])
summary

,value
SPECTRA AUC,0.879507
CL Top1,0.846154
CL Top2,0.934066
CL Top3,0.983516
